In this notebook, I will provide an overview of the dataset, perform cleaning and formatting based on the assumption checks conducted in 02_assumptions.ipynb, and saving it into a parquet format prior to performing the exploratory data analysis (EDA).

In [1]:
import pandas as pd
import os
import seaborn as sns 
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

In [2]:
df_raw = pd.read_csv("../data/raw/ncr_ride_bookings.csv")


In [3]:
df_raw.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [4]:
df_raw.info(memory_usage = "deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

I have a list of to-do's from the previous notebook:
 
1. Convert names into snake case
2. Map target variable 
3. Remove data leakage
4. Format Booking ID and Customer ID
5. Recast columns 


## Convert names into snake case

In [5]:
import re 
df_columns_clean = df_raw.copy()
def to_snake_case(df):
    new_columns = []
    for col in df.columns:
        col = col.strip()
        col = re.sub(r'[\s\-]+', '_', col)
        col = re.sub(r'[^\w_]', '', col)
        col = col.lower()
        new_columns.append(col)
    
    df.columns = new_columns
    return df

df_columns_clean = to_snake_case(df_columns_clean)
print(df_columns_clean.columns.tolist())

['date', 'time', 'booking_id', 'booking_status', 'customer_id', 'vehicle_type', 'pickup_location', 'drop_location', 'avg_vtat', 'avg_ctat', 'cancelled_rides_by_customer', 'reason_for_cancelling_by_customer', 'cancelled_rides_by_driver', 'driver_cancellation_reason', 'incomplete_rides', 'incomplete_rides_reason', 'booking_value', 'ride_distance', 'driver_ratings', 'customer_rating', 'payment_method']


## Map target variable

In [6]:
df_target_mapped = df_columns_clean.copy()

booking_mapping = {
    "Completed": 0,
    "Cancelled by Driver": 1,
    "No Driver Found": 1,
    "Cancelled by Customer": 1,
    "Incomplete": 0
}

df_target_mapped['is_cancelled'] = df_target_mapped['booking_status'].map(booking_mapping)
del df_target_mapped["booking_status"]
df_target_mapped['is_cancelled'].value_counts()

is_cancelled
0    102000
1     48000
Name: count, dtype: int64

## Remove data leakage

Leaking columns: 
- Cancelled Rides by Driver
- Cancelled Rides by Customer
- Reason for cancelling by Customer
- Driver Cancellation Reason
- Incomplete Rides
- Incomplete Rides Reason
- Driver Ratings
- Customer Rating

In [7]:
df_leak_clean = df_target_mapped.copy()

df_leak_clean.drop(columns = ["cancelled_rides_by_customer",
                                "reason_for_cancelling_by_customer",
                                "cancelled_rides_by_driver",
                                "driver_cancellation_reason",
                                "incomplete_rides",
                                "incomplete_rides_reason",
                                "driver_ratings",
                                "customer_rating"], inplace = True)

df_leak_clean.shape[0], df_leak_clean.shape[1]

(150000, 13)

## Format Booking ID and Customer ID

In [8]:
df_clean_quotes = df_leak_clean.copy()

cols_with_quotes = ["booking_id", 
                    "customer_id"]

for col in cols_with_quotes:
    df_clean_quotes[col] = df_clean_quotes[col].str.strip('"')

df_clean_quotes[["booking_id", 
                    "customer_id"]]

,booking_id,customer_id
0,CNR5884300,CID1982111
1,CNR1326809,CID4604802
2,CNR8494506,CID9202816
3,CNR8906825,CID2610914
4,CNR1950162,CID9933542
...,...,...
149995,CNR6500631,CID4337371
149996,CNR2468611,CID2325623
149997,CNR6358306,CID9925486
149998,CNR3030099,CID9415487


## Recast columns

In [9]:
df_typed = df_clean_quotes.copy()

df_typed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   date             150000 non-null  object 
 1   time             150000 non-null  object 
 2   booking_id       150000 non-null  object 
 3   customer_id      150000 non-null  object 
 4   vehicle_type     150000 non-null  object 
 5   pickup_location  150000 non-null  object 
 6   drop_location    150000 non-null  object 
 7   avg_vtat         139500 non-null  float64
 8   avg_ctat         102000 non-null  float64
 9   booking_value    102000 non-null  float64
 10  ride_distance    102000 non-null  float64
 11  payment_method   102000 non-null  object 
 12  is_cancelled     150000 non-null  int64  
dtypes: float64(4), int64(1), object(8)
memory usage: 14.9+ MB


### Temporal features

Converting column `date` and `time` would add unnecessary complexity so I will merge them into datetime later. 

### Categorical
1. Columns with "object" data type and no nulls are directly turned into Category
   
2. Columns with "object" data type and nulls are converted based on threshold. Threshold is set to 0.5, that means less than 50% of values are unique. If threshold is less that this value, the column will be turned into string[pyarrow] type. 

In [10]:
cat_ambiguous = ["booking_id", 
               "customer_id" ]
cat_threshold = 0.5

for col in cat_ambiguous:
    if col in df_typed.columns:
        num_unique = df_typed[col].nunique(dropna = False)
        ratio_unique = num_unique/len(df_typed)

        if ratio_unique <= cat_threshold:
            df_typed[col] = df_typed[col].astype("category")
            print(f"- Column {col} has been transformed into Category")
            print(f"Ratio of the column: {ratio_unique:.4%}")
        else:
            df_typed[col] = df_typed[col].astype("string[pyarrow]")
            print(f"- Column {col} has been transformed into string[pyarrow]")
            print(f"Ratio of the column: {ratio_unique:.4%}")


- Column booking_id has been transformed into string[pyarrow]
Ratio of the column: 99.1780%
- Column customer_id has been transformed into string[pyarrow]
Ratio of the column: 99.1920%


In [11]:
cat_columns = ["vehicle_type", 
               "pickup_location", 
               "drop_location",
               "payment_method"]

for col in cat_columns:
    if col in df_typed.columns:
        df_typed[col] = df_typed[col].astype("category")
        print(f"- Column {col} has been transformed into Category")

- Column vehicle_type has been transformed into Category
- Column pickup_location has been transformed into Category
- Column drop_location has been transformed into Category
- Column payment_method has been transformed into Category


### Numerical

i usually perform type optimization since I'm working on my personal computer. In this case, since the dataset is not very big, I will only transform the numerical data into float32

In [12]:
numerical_columns = ["avg_vtat", 
                 "avg_ctat", 
                 "booking_value",
                 "ride_distance",
                 "is_cancelled"]

for col in numerical_columns:
    if col in df_typed.columns:
        df_typed[col] = df_typed[col].astype("float32")
        print(f"- Column {col} has been transformed into float32")


- Column avg_vtat has been transformed into float32
- Column avg_ctat has been transformed into float32
- Column booking_value has been transformed into float32
- Column ride_distance has been transformed into float32
- Column is_cancelled has been transformed into float32


## Train/test split

The dataset goes from the 1st of January 2024 to the 30th December 2024, I will split it following 75% train, 15% val, 10% test.

In [13]:
df_typed = df_typed.sort_values("date")

train_end = "2024-09-30"  
val_end = "2024-11-30"

train_df = df_typed[df_typed["date"] <= train_end]
val_df   = df_typed[(df_typed["date"] > train_end) & (df_typed["date"] <= val_end)]
test_df  = df_typed[df_typed["date"] > val_end]

In [14]:
print(f" Train: {len(train_df)}")
print(f" Validation : {len(val_df)}")
print(f" Test: {len(test_df)}")

 Train: 112705
 Validation : 25045
 Test: 12250


## Save dataset splits

In [15]:
output_dir = "../data/bronze"
os.makedirs(output_dir, exist_ok=True)  

df_typed.to_parquet(os.path.join(output_dir, "clean_dataset.parquet"),  engine='pyarrow', index=False)
train_df.to_parquet(os.path.join(output_dir, "train.parquet"), engine='pyarrow', index=False)
val_df.to_parquet(os.path.join(output_dir, "val.parquet"), engine='pyarrow', index=False)
test_df.to_parquet(os.path.join(output_dir, "test.parquet"), engine='pyarrow', index=False)